# NumiNaMath Dataset Evaluation Notebook

This notebook demonstrates how to:
1. Load the NumiNaMath dataset from local files or HuggingFace.
2. Preprocess samples to extract the problem and boxed solution.
3. Call a frontier model API (e.g., GPT) using an API key from environment variables.
4. Run inference on 5 random test samples.
5. Compute cross-entropy between ground truth and prediction.


## 1. Import Required Libraries

We will use `datasets` for data loading, `re` for regex extraction, `os` for environment variables, and `random` for sampling.

In [1]:
import os
from pathlib import Path

#------------ CACHE ENVIRONMENT ----------------------------------------
# Set cache environment variables BEFORE importing datasets
# or cache env variables will be reset at import and trigger errors
PROJECT_ROOT = Path.cwd().parent  # Go up one level
hf_cache = PROJECT_ROOT / ".cache" / "huggingface"
hf_cache.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(hf_cache)
os.environ["HF_DATASETS_CACHE"] = str(hf_cache / "datasets")
os.environ["HF_HUB_CACHE"] = str(hf_cache)
os.environ["TRANSFORMERS_CACHE"] = str(hf_cache / "transformers")

In [2]:

# -------------- other imports -----------------------------------------
import re
import random
from datasets import load_dataset, Dataset, load_from_disk
import requests
import numpy as np
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig
import torch
from pprint import pprint

## 2. Load the NumiNaMath Dataset

Try to load the dataset from the local directory. If not found, download from HuggingFace.

In [3]:
# Define local dataset path
local_dataset_path = Path('../datasets/test/data-00000-of-00001.arrow').resolve()

# dataset ID
HF_DATASET_ID = "AI-MO/NuminaMath-TIR"
PROJECT_ROOT = Path.cwd().parent  # Go up one level
DATASET_DIR = PROJECT_ROOT / "datasets"

# logic
if local_dataset_path.exists():
    print(f"Loading dataset from local path: {local_dataset_path}")
    # dataset = Dataset.load_from_disk(str(local_dataset_path.parent.parent))
    dataset = load_from_disk(str(DATASET_DIR))
    test_split = dataset['test']
else:
    print("Local dataset not found. Downloading from HuggingFace...")
    dataset = load_dataset(HF_DATASET_ID)
    test_split = dataset['test']

print(f"Loaded {len(test_split)} test samples.")

Loading dataset from local path: /home/benjamin.deporte/GenAI_Math/datasets/test/data-00000-of-00001.arrow
Loaded 99 test samples.


In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['problem', 'solution', 'messages'],
        num_rows: 72441
    })
    test: Dataset({
        features: ['problem', 'solution', 'messages'],
        num_rows: 99
    })
})

In [5]:
train_dataset = dataset['train'].select(range(int(0.01*len(dataset['train']))))
test_dataset = dataset['test']

print(f"Train size:          {len(train_dataset)}")
print(f"Validation size:     {len(test_dataset)}")

Train size:          724
Validation size:     99


In [6]:
train_dataset

Dataset({
    features: ['problem', 'solution', 'messages'],
    num_rows: 724
})

In [7]:
example = train_dataset[0]
pprint(example)

{'messages': [{'content': 'What is the coefficient of $x^2y^6$ in the '
                          'expansion of '
                          '$\\left(\\frac{3}{5}x-\\frac{y}{2}\\right)^8$?  '
                          'Express your answer as a common fraction.',
               'role': 'user'},
              {'content': 'To determine the coefficient of \\(x^2y^6\\) in the '
                          'expansion of \\(\\left(\\frac{3}{5}x - '
                          '\\frac{y}{2}\\right)^8\\), we can use the binomial '
                          'theorem.\n'
                          '\n'
                          'The binomial theorem states:\n'
                          '\\[\n'
                          '(a + b)^n = \\sum_{k=0}^{n} \\binom{n}{k} a^{n-k} '
                          'b^k\n'
                          '\\]\n'
                          '\n'
                          'In this case, \\(a = \\frac{3}{5}x\\), \\(b = '
                          '-\\frac{y}{2}\\), and \\(n = 8\\).

In [8]:
# first utility function,
# extract the string between the opening \boxed{ and the corresponding closing }
def extract_boxed(solution: str) -> str:
    """ 
    count the opening { and closing } to extract the right string
    """
    start = solution.find(r'\boxed{')
    if start == -1:
        # raise ValueError("Boxed solution not found in solution field.")*
        return ""
        
    # Move to the opening brace
    brace_pos = start + len(r'\boxed{') - 1
    depth = 0
    for i in range(brace_pos, len(solution)):
        if solution[i] == '{':
            depth += 1
        elif solution[i] == '}':
            depth -= 1
            if depth == 0:
                return solution[brace_pos + 1:i]  # content between the braces
    raise ValueError("Unmatched braces in \\boxed{} expression.")

## utility function

def extract_problem_and_boxed_solution(sample):
    """
    Utility function to extract problem, solution, and boxed solution from a sample in the test split of NuminaMath
    Inputs :
        - sample (dict, with keys:
            'problem' : str,
            'solution' : str,
            'messages' : list of dict,
        }
        the value sample['solution'] contains the chain of thoughts and the ultimate solution in \boxed{ .. }
    Returns:
        - problem (str) : the problem
        - solution (str) : the whole ground truth response
        - boxed_solution (str) : the extract between \boxed{ and }, if any, None otherwise
    """
    
    problem = sample.get('problem', None)
    solution = sample.get('solution', None)
    if problem is None or solution is None:
        raise ValueError("Sample missing 'problem' or 'solution' field.")
    
    # Match everything between \\boxed{ and the corresponding closing }
    boxed_solution = extract_boxed(solution)
    
    return problem, solution, boxed_solution

In [9]:
# Preprocess the dataset pour avoir le format qui va bien pour le SFT

def preprocess_function(example):
    
    problem, solution, boxed_solution = extract_problem_and_boxed_solution(example)
    
    return {
        "prompt": [{"role": "user", "content": problem}],
        "completion": [
            {"role": "assistant", "content": f"<think>{solution}</think><answer>{boxed_solution}</answer>"}
        ],
    }

In [10]:
preprocessed_train_dataset = train_dataset.map(preprocess_function, remove_columns=["problem", "solution", "messages"])
preprocessed_test_dataset = test_dataset.map(preprocess_function, remove_columns=["problem", "solution", "messages"])

pprint(next(iter(preprocessed_train_dataset)))

Map:   0%|          | 0/724 [00:00<?, ? examples/s]

{'completion': [{'content': '<think>To determine the coefficient of '
                            '\\(x^2y^6\\) in the expansion of '
                            '\\(\\left(\\frac{3}{5}x - '
                            '\\frac{y}{2}\\right)^8\\), we can use the '
                            'binomial theorem.\n'
                            '\n'
                            'The binomial theorem states:\n'
                            '\\[\n'
                            '(a + b)^n = \\sum_{k=0}^{n} \\binom{n}{k} a^{n-k} '
                            'b^k\n'
                            '\\]\n'
                            '\n'
                            'In this case, \\(a = \\frac{3}{5}x\\), \\(b = '
                            '-\\frac{y}{2}\\), and \\(n = 8\\).\n'
                            '\n'
                            'We are interested in the term that contains '
                            '\\(x^2y^6\\). In the general term of the binomial '
                            'expansio

# 3. Load model, perform LoRA

In [11]:
# utility code - idem, télécharge le modèle si pas déjà présent sur le disque
# et le sauve localement

MODEL_ID = "Qwen/Qwen3-0.6B"
MODEL_DIR = Path("../models/qwen3-0.6B-SFT").resolve()

MODEL_DIR.mkdir(parents=True, exist_ok=True)
is_empty = not any(MODEL_DIR.iterdir())

if is_empty:
    print(f"{MODEL_DIR} is empty -> downloading model/tokenizer from Hub and saving to disk...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,   # ou torch.float16 selon ton GPU
        device_map="auto"
    )
    tokenizer.save_pretrained(str(MODEL_DIR))
    model.save_pretrained(str(MODEL_DIR))
else:
    print(f"{MODEL_DIR} is not empty -> loading model/tokenizer from disk...")
    tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(
        str(MODEL_DIR),
        local_files_only=True,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

/home/benjamin.deporte/GenAI_Math/models/qwen3-0.6B-SFT is not empty -> loading model/tokenizer from disk...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [12]:
# utility function pour calculer le nombre de paramètres trainables
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, total

In [13]:
base_model = model

# Before LoRA (full model)
full_trainable, full_total = count_params(base_model)

print(f"Modèle : {MODEL_ID}, full FT trainable: {full_trainable:,}")

Modèle : Qwen/Qwen3-0.6B, full FT trainable: 596,049,920


In [14]:
# using PEFT and LoRA pour réduire le besoin en VRAM
RANG=4

peft_config=LoraConfig(
    task_type="CAUSAL_LM",   # decrit quel type de modèle on adapte
    inference_mode=False, # ordonne que les paramètres soient entrainables
    r=RANG, # rang !
    lora_alpha=2*RANG, # scaling factor pour l'update LoRA
    lora_dropout=0.05, # dropout pour le LoRA
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # LoRA sur les têtes d'attention
)

lora_model=get_peft_model(base_model, peft_config)

In [15]:
# After LoRA wrapping
lora_trainable, lora_total = count_params(lora_model)

reduction_vs_full_ft = 100 * (1 - (lora_trainable / full_trainable))
trainable_pct_of_total = 100 * (lora_trainable / lora_total)

print(f"Modèle : {MODEL_ID}, full FT trainable: {full_trainable:,}")
print(f"LoRA trainable:    {lora_trainable:,}")
print(f"Reduction:         {reduction_vs_full_ft:.2f}%")
print(f"LoRA trainable %:  {trainable_pct_of_total:.4f}%")

Modèle : Qwen/Qwen3-0.6B, full FT trainable: 596,049,920
LoRA trainable:    1,146,880
Reduction:         99.81%
LoRA trainable %:  0.1920%


# 4. Evaluation metrics

In [16]:
# Optional metric: token-level accuracy (ignores -100 labels)
def preprocess_logits_for_metrics(logits, labels):
    # logits est typiquement (batch, seq_len, vocab_size)
    if isinstance(logits, tuple):
        logits = logits[0]
    # renvoie (batch, seq_len) avec le predicted token id
    return logits.argmax(dim=-1)

def compute_metrics(eval_pred):
    preds, labels = eval_pred  # numpy arrays
    # next-token shift
    preds = preds[:, :-1].reshape(-1)
    labels = labels[:, 1:].reshape(-1)

    mask = labels != -100
    if mask.sum() == 0:
        return {"token_accuracy": 0.0}

    # compte 1 pour les bons tokens prédits, et la moyenne
    acc = (preds[mask] == labels[mask]).astype(np.float32).mean().item()
    return {"token_accuracy": acc}

# Training

In [17]:
OUTPUT_DIR = PROJECT_ROOT / "outputs/sft-numinamath"
LOGGING_DIR = PROJECT_ROOT / "logs"

# Enable eval during training
sft_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,      # important pour ma pauvre 3080 Ti 12 Go
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    num_train_epochs=3,
    
    # logging strategy
    logging_strategy="epoch",
    logging_dir=str(LOGGING_DIR),
    # logging_steps=10,
    # max_seq_length=512,
    gradient_checkpointing=True,
    bf16=True,   
    
    # evaluation strategy
    eval_strategy="steps",             # if your version expects evaluation_strategy, use that key instead
    eval_steps=100,
    
    # save model strategy
    save_steps=100,
    save_strategy="steps",
    
    # other
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    eval_accumulation_steps=8,         # helps eval memory
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [18]:
trainer = SFTTrainer(
    model=lora_model,  # use the already reduced model
    args=sft_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    # processing_class=tokenizer
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics
)

Tokenizing train dataset:   0%|          | 0/724 [00:00<?, ? examples/s]

In [19]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Token Accuracy
100,0.604569,0.572396,0.830524
138,0.604569,0.569240,0.831507


TrainOutput(global_step=138, training_loss=0.6405479320581409, metrics={'train_runtime': 589.0498, 'train_samples_per_second': 3.687, 'train_steps_per_second': 0.234, 'total_flos': 3722718763155456.0, 'train_loss': 0.6405479320581409})

In [ ]:
model_for_eval = trainer.model
model_for_eval.eval()

# Evaluation

In [ ]:
def parse_completion(completion_text):
    """
    Parse completion text to extract reasoning and final answer.
    Format: <think>REASONING</think><answer>FINAL_ANSWER</answer>
    
    Returns: (reasoning_text, final_answer_text)
    """
    if "<think>" in completion_text and "</think>" in completion_text:
        start_idx = completion_text.find("<think>") + len("<think>")
        end_idx = completion_text.find("</think>")
        reasoning = completion_text[start_idx:end_idx].strip()
        if "<answer>" in completion_text and "</answer>" in completion_text:
            start_idx = completion_text.find("<answer>") + len("<answer>")
            end_idx = completion_text.find("</answer>")
            final_answer = completion_text[end_idx + len("</answer>"):].strip()
    else:
        # No think tags, treat entire text as final answer
        reasoning = ""
        final_answer = "" # completion_text.strip()
    
    return reasoning, final_answer

In [ ]:
N_SAMPLES = 5
SAMPLE_SEED = 123
MAX_NEW_TOKENS = 2048

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_for_eval = trainer.model
model_for_eval.eval()

rng = np.random.default_rng(SAMPLE_SEED)
n_eval = min(N_SAMPLES, len(test_dataset))
sample_indices = rng.choice(len(test_dataset), size=n_eval, replace=False).tolist()

print(f"Sampling {n_eval} examples from validation set of size {len(test_dataset)}")

In [ ]:
results = []
sampled_indices = random.sample(range(len(test_split)), 3)

for idx in sampled_indices:
    sample = test_split[idx]
    try:
        problem, solution, boxed_solution = extract_problem_and_boxed_solution(sample)
    except ValueError as e:
        print(f"Skipping sample {idx}: {e}")
        continue
    
    results.append({
        'problem': problem,
        'solution': solution,
        'ground_truth': boxed_solution,
    })
    
for i, res in enumerate(results):
    print(f"sample {i+1}", "-"*40)
    print(f"problem: {res['problem']}")
    print(f"full solution: {res['solution']}")
    print(f"ground truth: {res['ground_truth']}\n\n")

# Run Inference on some Random Test Samples

Randomly select some samples, extract the problem and boxed solution, call the model, and collect predictions.

In [ ]:
N_SAMPLES=5

results = []
sampled_indices = random.sample(range(len(test_split)), N_SAMPLES)

for idx in sampled_indices:
    sample = test_split[idx]
    try:
        problem, solution, boxed_solution = extract_problem_and_boxed_solution(sample)
    except ValueError as e:
        print(f"Skipping sample {idx}: {e}")
        continue
    
    try:
        query = {
            "prompt": [{"role": "user", "content": problem}],
            "completion": [
                {"role": "assistant", "content": f"<think>{solution}</think><answer>{boxed_solution}</answer>"}
                ],
            }    
        prompt_text = tokenizer.apply_chat_template(
            query,
            tokenize=False,
            add_generation_prompt=True,
            )
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model_for_eval.device)
        with torch.no_grad():
            generated = model_for_eval.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
            
        prompt_len = inputs["input_ids"].shape[1]
        gen_tokens = generated[0][prompt_len:]
        pred_completion = tokenizer.decode(gen_tokens, skip_special_tokens=True)

        # Parse ground truth
        gt_full = ex["completion"][0]["content"]
        gt_reasoning, gt_final_answer = parse_completion(gt_full)

        # Parse prediction
        pred_reasoning, pred_final_answer = parse_completion(pred_completion)

        results.append(
            {
                "idx": int(idx),
                "prompt": messages[0]["content"],
                "gt_reasoning": gt_reasoning,
                "gt_final_answer": gt_final_answer,
                "pred_reasoning": pred_reasoning,
                "pred_final_answer": pred_final_answer,
                "gt_full": gt_full,
                "pred_full": pred_completion,
            }
        )
    except ValueError as e:
        print(f"Generation failed")
        continue

print(f"Built {len(results)} comparisons with parsed components.")

for i, res in enumerate(results):
    print(f"Sample {i+1}:")
    print(f"Problem: {res['problem']}")
    print(f"Full solution: {res['solution'][-100:]}")
    print(f"Ground Truth: {res['ground_truth']}")
    print(f"Full answer: {res['full_answer']}")
    print(f"Prediction: {res['prediction']}")
    print('-'*40, "\n")

# Comparing ground truth and predictions

In [ ]:
for i, res in enumerate(results):
    print(f"Sample {i+1}:")
    print(f"Ground Truth: {res['ground_truth']}")
    print(f"Prediction: {res['prediction']}")
    print('-'*40, "\n")

In [ ]:
import os
print("HF_HOME:", os.environ.get("HF_HOME"))
print("HF_HUB_CACHE:", os.environ.get("HF_HUB_CACHE"))
print("TRANSFORMERS_CACHE:", os.environ.get("TRANSFORMERS_CACHE"))

In [ ]:
# First attempt : use BERTScore

from bert_score import score

for i, res in enumerate(results):
    print(f"Sample {i+1}:")
    gt = res['ground_truth']
    pred = res['prediction']
    print(f"Ground Truth: {gt}")
    print(f"Prediction: {pred}")
    P, R, F1 = score([gt], [pred], lang="en")
    print(f"P = {P.item():.3f}, R = {R.item():.3f}, F1 = {F1.item():.3f}")
    print('-'*40, "\n")